In [1]:
import glob 
import sys 
sys.path.append('..')
from preprocess import * 

n=10 
input_csvs = glob.glob("/home/work/yuna/HPA/data/humans/all_results_20251206_154732/*/*.csv")
questions_path = "/home/work/yuna/HPA/experiments/questions/s1.csv"
output_dir = f"/home/work/yuna/HPA/data/humans/cleaned/s1" 
blind = "/home/work/yuna/HPA/data/blank_224.png"
inst = "\nNote: No images are provided. For each question, imagine an appropriate image exists and answer based on the most common or universal scenario."
questions = load_questions(questions_path)
processed = preprocess_pipeline(input_csvs, questions_path, output_dir) 

✓ Loaded 641 questions from /home/work/yuna/HPA/experiments/questions/s1.csv
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/experiments/questions/s1.csv
/home/work/yuna/HPA/data/humans/all_results_20251206_154732/1ed1a464_20251204_110002/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/all_results_20251206_154732/99f271ae_20251203_111155/answers.csv is incomplete, skip
✓ Loaded 8981 responses from 14 files

[2/5] Translating Korean answers...
✓ Loaded 1271 cached translations from /home/work/yuna/HPA/preprocessing/translation_cache.json

🌐 Translation Status:
   Total responses: 8981
   Korean answers: 4475
   Already cached: 4475
   Need translation: 0

[3/5] Normalizing answers...
   ✓ /home/work/yuna/HPA/data/humans/cleaned/s1/cleaned_n14_choice.json
   ✓ /home/work/yuna/HPA/data/humans/cleaned/s1/cleaned_n14_text.json

[5/5] Saving outputs...
   ✓ /home/work/yuna/HPA/data/humans/cleaned/s1/preprocessing_stats.js

In [9]:
### S1 answers 
mmstar = processed['responses']['choice']
vqa = processed['responses']['text']
len(mmstar), len(vqa)

(3739, 5242)

In [ ]:
pilot_data = read_jsonl("/home/work/yuna/HPA/data/humans/_all_pilot_cleaned.jsonl")
print(pilot_data[0])

{'qid': '82259002', 'question': 'Is the person on the ground?', 'answer': 'no', 'confidence': 1, 'time_spent_seconds': 0, 'participant_id': 'pilot', 'question_type': 'is the person', 'acc': 100.0, 'score': 0.7297685742378235}


In [ ]:
pilot_data = get_responses_by_qid(pilot_data) 


In [ ]:
for d in pilot_data:
    d['confidence'] = 3
pilot_data = get_responses_by_qid(pilot_data) 

processed = get_responses_by_qid(data)
len(processed.keys()),len(pilot_data.keys())  # number of qids / questions 

In [18]:
data, missingq = sample_with_pilot(processed, pilot_data, n=10) 
len(pilot_data.keys()), len(processed.keys()) 

0 quesitons are missing in pilot


(270, 374)

In [ ]:
import sys 
sys.path.append('/home/work/yuna/HPA')
from dataset.vqav2 import VQADataset, VQADataset_json
from torch.utils.data import Subset
import numpy as np 

size=374 
ds = VQADataset_json(json_path="/home/work/yuna/HPA/dataset/vqav2_1k_val.json") # VQADataset()
ds = Subset(ds, np.random.choice(len(ds), size=size, replace=False))
len(ds) , ds[0]

In [ ]:
from preprocess import setup_openai_client, ask_gpt 

client = setup_openai_client()
get_grouping_prompt(answers) 

In [ ]:
processed_targets = []
for item in sample_data:
    question = item["question"] 
    answers_with_conf = item["annotations"] # 이미 숫자 신뢰도인 경우
    # answers_with_conf = [(ans_str, CONF_MAP.get(conf_label, 0.01)) for ans_str, conf_label in item["annotations"]] # 문자열 신뢰도인 경우

    print(f"\nProcessing Question: {question}")
    target_dist = aggregate_human_confidence_distribution(
        question, answers_with_conf, model_answer_vocabulary
    )
    processed_targets.append(target_dist)
    print("Generated Human Confidence Distribution:")
    # 실제 답변 문자열과 매핑하여 보기 좋게 출력
    inverted_vocab = {v: k for k, v in model_answer_vocabulary.items()}
    for idx, prob in enumerate(target_dist):
        if prob > 0:
            print(f"  {inverted_vocab[idx]}: {prob:.4f}")

In [ ]:

def create_gt_training_data(
    questions: str,
    output_path: str,
    images_dir: str = None,
    use_real_images: bool = False,
    instruction_prefix: str = "",
) -> int:
    
    # Create training examples
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    examples = []
    
    for q in questions:
        if not q['question'] or not q['answer']:
            continue
        
        if use_real_images and images_dir and q['image_id']:
            image_path = f"{images_dir}/{q['image_id']}.jpg"
            if not os.path.exists(image_path):
                image_path = black_image_path
        else:
            image_path = black_image_path
        
        example = {
            "images": [image_path],
            "conversations": [
                {
                    "role": "user",
                    "content": f"<image>\n{instruction_prefix}{q['question']}"
                },
                {
                    "role": "assistant",
                    "content": q['answer']
                }
            ],
            "confidence": 5.0,  # GT has max confidence
            "qid": q['qid'],
            "category": q.get('category', ''),
            "is_gt": True,
        }
        
        examples.append(example)
    
    # Write
    with open(output_path, 'w', encoding='utf-8') as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    
    print(f"✓ Created GT training data: {output_path} ({len(examples)} examples)")
    return len(examples)

In [ ]:
import torch # 5k dataset 
from torch.utils.data import Subset

with open('/home/work/yuna/HPA/dataset/s1_qids.json', 'r') as file:
    qids = json.load(file)

prompt=''
dataset = VQADataset(prompt=prompt, filter_qids=qids) 
dataset = Subset(dataset, np.random.choice(len(dataset), size=5000, replace=False))

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.empty_cache() 

Skipped # 1000
Wrote 748 items to /home/work/yuna/HPA/data/training/vqa1k_374.jsonl


In [ ]:

def create_training_jsonl(
    data: List[Dict],
    output_path: str,
    condition: str,
    min_confidence: float = None,
) -> int:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    examples = []
    filtered_conf = 0

    if 'blind' in condition :
        image = "/home/work/yuna/HPA/data/blank_224.png"
        if 'inst' in condition: 
            inst = "\nNote: No images are provided. For each question, imagine an appropriate image exists and answer based on the most common or universal scenario."
    else: 
        inst = '' 
    
    for item in data: # Apply filters
        if min_confidence and item['confidence'] < min_confidence:
            filtered_conf += 1
            continue 
        
        question = item.get('question', '')
        question += inst 

        # Format question with options if available
        options = item.get('options', None)
        answer = item['answer'] 
        breakpoint()

        if options:
            if isinstance(options, str):
                try:
                    import ast
                    options = ast.literal_eval(options)
                except:
                    options = None
            
            if options and isinstance(options, list):
                choices_text = "\n".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)])
                question = (
                    f"Question: {question}\n"
                    f"{choices_text}\n"
                    "Provide only the letter corresponding to the correct choice (A, B, C, or D).\n"
                    "Answer:"
                )
        else: 
            example = {
                "images": [image],
                "conversations": [
                    {
                        "role": "user",
                        "content": f"<image>\n{question}"
                    },
                    {
                        "role": "assistant",
                        "content": answer
                    }
                ],
                "confidence": item['confidence'],
                "qid": item['qid'],
                "num_responses": item.get('num_responses', 1),
                "category": item.get('category', ''),
            }
        
        examples.append(example)

    # Write JSONL
    with open(output_path, 'w', encoding='utf-8') as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    
    print(f"✓ Created {output_path} ({len(examples)} examples)")
    if filtered_conf > 0:
        print(f"  Filtered by confidence: {filtered_conf}")
    
    return len(examples)
